In [ ]:
!pip install -q openai python-dotenv pillow IPython

In [ ]:
import os
import json
import sqlite3

from openai import OpenAI
from IPython.display import Audio, SVG, display

# ----------------------------------------------------
# Add your OpenAI API Key
# ----------------------------------------------------

client = OpenAI(
    api_key="YOUR_OPENAI_API_KEY"
)

In [ ]:
# ----------------------------------------------------
# Tiny SQLite database containing travel information
# ----------------------------------------------------

conn = sqlite3.connect("travel.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS destinations(
    city TEXT,
    attraction TEXT,
    days INTEGER
)
""")

cursor.execute("DELETE FROM destinations")

rows = [

    ("Goa","Baga Beach",3),
    ("Goa","Dudhsagar Falls",1),
    ("Goa","Fort Aguada",1),

    ("Jaipur","Amber Fort",2),
    ("Jaipur","City Palace",1),

    ("Delhi","India Gate",1),
    ("Delhi","Red Fort",2),

    ("Hyderabad","Charminar",1),
    ("Hyderabad","Golconda Fort",2)

]

cursor.executemany(
    "INSERT INTO destinations VALUES(?,?,?)",
    rows
)

conn.commit()

print("Database Ready.")

In [ ]:
# ----------------------------------------------------
# Tiny RAG Knowledge Base
# ----------------------------------------------------

knowledge_base = [

"""
Goa is famous for beaches, seafood, nightlife and Portuguese architecture.
""",

"""
Jaipur is called the Pink City and is famous for forts and royal palaces.
""",

"""
Delhi has numerous historical monuments including Red Fort and India Gate.
""",

"""
Hyderabad is famous for Charminar, Golconda Fort and Hyderabadi Biryani.
"""

]


def retrieve_context(query):

    query = query.lower()

    retrieved = []

    for doc in knowledge_base:

        if any(word in doc.lower() for word in query.split()):
            retrieved.append(doc)

    return "\n".join(retrieved)

In [ ]:
# ----------------------------------------------------
# Tool 1
# Search SQLite
# ----------------------------------------------------

def search_destination(city):

    cur = conn.cursor()

    cur.execute(
        "SELECT attraction, days FROM destinations WHERE city=?",
        (city,)
    )

    result = cur.fetchall()

    return result


# ----------------------------------------------------
# Tool 2
# Generate itinerary
# ----------------------------------------------------

def generate_itinerary(city):

    places = search_destination(city)

    if len(places) == 0:
        return "No itinerary available."

    itinerary = f"\nTravel Plan for {city}\n\n"

    day = 1

    for place, duration in places:

        itinerary += f"Day {day}: Visit {place}\n"

        day += duration

    return itinerary


# ----------------------------------------------------
# Tool 3
# Retrieve RAG Context
# ----------------------------------------------------

def rag_search(query):

    context = retrieve_context(query)

    if context == "":
        return "No additional information."

    return context


# ----------------------------------------------------
# Tool Registry
# ----------------------------------------------------

TOOLS = {

    "generate_itinerary": generate_itinerary,

    "rag_search": rag_search

}

print("Tools Ready.")

In [ ]:
# -------------------------------------------------------
# Chat Assistant + Tool Calling
# -------------------------------------------------------

tools = [
    {
        "type": "function",
        "name": "generate_itinerary",
        "description": "Generate a travel itinerary for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string"
                }
            },
            "required": ["city"]
        }
    },
    {
        "type": "function",
        "name": "rag_search",
        "description": "Retrieve travel knowledge for a user query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string"
                }
            },
            "required": ["query"]
        }
    }
]


def travel_assistant(user_query):

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=user_query,
        tools=tools
    )

    messages = []

    for item in response.output:

        if item.type == "function_call":

            tool_name = item.name

            arguments = json.loads(item.arguments)

            print(f"Calling Tool -> {tool_name}")

            tool_result = TOOLS[tool_name](**arguments)

            messages.append(item)

            messages.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": tool_result
            })

    if len(messages):

        final = client.responses.create(
            model="gpt-4.1-mini",
            input=messages
        )

        return final.output_text

    return response.output_text

In [ ]:
question = """
Plan a 3-day Goa vacation.
Use your tools whenever needed.
"""

answer = travel_assistant(question)

print(answer)

In [ ]:
# -------------------------------------------------------
# Image Generation
# -------------------------------------------------------

result = client.images.generate(
    model="gpt-image-1",
    prompt="""
Create a vibrant travel poster for Goa.

Include

- Beach
- Palm Trees
- Sunset
- Tourists
- Beautiful typography
- Premium travel brochure style
"""
)

image_base64 = result.data[0].b64_json

import base64

with open("goa_poster.png", "wb") as f:
    f.write(base64.b64decode(image_base64))

print("Poster Saved.")

display(Image(filename="goa_poster.png"))

In [ ]:
# -------------------------------------------------------
# SVG Generation using Python
# -------------------------------------------------------

svg = """
<svg width="400" height="220"
xmlns="http://www.w3.org/2000/svg">

<rect
x="5"
y="5"
width="390"
height="210"
fill="#E8F6FF"
stroke="#1E88E5"
stroke-width="3"/>

<text
x="25"
y="40"
font-size="26"
font-weight="bold"
fill="#0D47A1">

✈ Boarding Pass

</text>

<line
x1="20"
y1="60"
x2="380"
y2="60"
stroke="gray"/>

<text x="20" y="95">
Passenger : Alex
</text>

<text x="20" y="125">
From : Hyderabad
</text>

<text x="20" y="155">
To : Goa
</text>

<text x="20" y="185">
Seat : A12
</text>

</svg>
"""

with open("boarding_pass.svg", "w") as f:
    f.write(svg)

display(SVG(filename="boarding_pass.svg"))

In [ ]:
# -------------------------------------------------------
# Generate SVG using GPT
# -------------------------------------------------------

svg_prompt = """
Generate ONLY valid SVG code.

Create a modern airline logo.

Requirements:
- Blue color theme
- Airplane icon
- Circular logo
- Clean minimalist design
- SVG only
"""

response = client.responses.create(
    model="gpt-4.1-mini",
    input=svg_prompt
)

svg_code = response.output_text

# Remove markdown fences if present
svg_code = svg_code.replace("```svg", "")
svg_code = svg_code.replace("```", "").strip()

with open("airline_logo.svg", "w", encoding="utf-8") as f:
    f.write(svg_code)

print("SVG Generated Successfully!")

display(SVG(filename="airline_logo.svg"))

In [ ]:
# -------------------------------------------------------
# OpenAI Text-to-Speech
# -------------------------------------------------------

speech_text = """
Welcome aboard!

Your Goa vacation has been successfully planned.

Enjoy your beaches, sightseeing, delicious seafood,
and have a wonderful journey.
"""

response = client.audio.speech.create(

    model="gpt-4o-mini-tts",

    voice="alloy",          # Try: alloy, ash, ballad, coral, echo, sage, shimmer

    input=speech_text
)

response.stream_to_file("travel_audio.mp3")

print("Audio Generated!")

In [ ]:
# -------------------------------------------------------
# Play Generated Audio
# -------------------------------------------------------

Audio("travel_audio.mp3")

In [ ]:
# -------------------------------------------------------
# Complete Multi-Modal Travel Assistant
# -------------------------------------------------------

city = input("Enter destination city: ")

print("=" * 60)
print("AI TRAVEL ASSISTANT")
print("=" * 60)

# -------------------------------
# Step 1
# Retrieve RAG Context
# -------------------------------

context = rag_search(city)

print("\nRetrieved Knowledge\n")
print(context)

# -------------------------------
# Step 2
# SQLite Itinerary
# -------------------------------

itinerary = generate_itinerary(city)

print("\nSuggested Itinerary\n")
print(itinerary)

# -------------------------------
# Step 3
# Ask LLM
# -------------------------------

prompt = f"""
You are an AI Travel Assistant.

Destination:
{city}

Knowledge:
{context}

Itinerary:
{itinerary}

Create a friendly travel guide.
"""

response = client.responses.create(
    model="gpt-4.1-mini",
    input=prompt
)

guide = response.output_text

print("\nTravel Guide\n")
print(guide)

# -------------------------------
# Step 4
# Generate Poster
# -------------------------------

print("\nGenerating Poster...")

image = client.images.generate(
    model="gpt-image-1",
    prompt=f"""
Luxury travel poster for {city}.

Highly realistic.

Travel magazine cover.

Vibrant colors.

Professional tourism advertisement.
"""
)

import base64

with open("travel_poster.png", "wb") as f:
    f.write(base64.b64decode(image.data[0].b64_json))

display(Image(filename="travel_poster.png"))

# -------------------------------
# Step 5
# Generate Audio Guide
# -------------------------------

print("\nGenerating Audio Guide...")

tts = client.audio.speech.create(

    model="gpt-4o-mini-tts",

    voice="sage",

    input=guide
)

tts.stream_to_file("guide.mp3")

Audio("guide.mp3")